# Build Embedding Dataset

Goal:
Convert selected audio samples into embeddings.

Input:
ESC50 audio

Output:
Embeddings + labels

Pipeline:

Audio
↓

YAMNet
↓

Embedding
↓

Training Dataset

In [ ]:
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import tensorflow_hub as hub
from tqdm import tqdm

In [ ]:
ROOT = Path("../data")

RAW = ROOT / "raw" / "esc50"

AUDIO = RAW / "audio"

META = RAW / "meta" / "esc50.csv"

PROCESSED = ROOT / "processed"

PROCESSED.mkdir(exist_ok=True)

In [ ]:
df = pd.read_csv(META)

In [ ]:
df.head()

## Select Aegis Sounds

In [ ]:
TARGET = {
    "alarm": ["clock_alarm"],
    "baby": ["crying_baby"],
    "knock": ["door_wood_knock"],
    "siren": ["siren"],
}

In [ ]:
selected = []

for values in TARGET.values():
    selected.extend(values)

In [ ]:
subset = df[df["category"].isin(selected)]

len(subset)

## Load Model

In [ ]:
model = hub.load("https://tfhub.dev/google/yamnet/1")

## Extract Embeddings

In [ ]:
X = []

y = []

meta = []

In [ ]:
for _, row in tqdm(subset.iterrows(), total=len(subset)):
    path = AUDIO / row["filename"]

    waveform, sr = librosa.load(path, sr=16000)

    waveform = waveform.astype(np.float32)

    scores, embeddings, spec = model(waveform)

    emb = embeddings.numpy().mean(axis=0)

    X.append(emb)

    y.append(row["category"])

    meta.append(row["filename"])

In [ ]:
X = np.array(X)

y = np.array(y)

print(X.shape)

print(y.shape)

## Save Processed Dataset

In [ ]:
np.save(PROCESSED / "X.npy", X)

np.save(PROCESSED / "y.npy", y)

In [ ]:
pd.DataFrame({"file": meta, "label": y}).to_csv(PROCESSED / "metadata.csv", index=False)

# Findings

Embedding extraction works.

Dataset generated.

Ready for training.